In [ ]:
"""
Created on Tue Apr  8 10:37:30 2025

@author: liuzzil2
"""

import numpy as np
import matplotlib.pyplot as plt
import mne
from mne_connectivity import spectral_connectivity_epochs
from mne_connectivity import envelope_correlation

## Define Presets

In [ ]:
fs = 600 # sampling frequency
t = 5 # total time in seconds
f = 50 # oscillation frequency in Hz
fa = 4 # amplitude modulation frequency
phi2 = np.deg2rad(90) # phase shift of oscillation 2
phi3 = np.deg2rad(10) # phase shift of oscillation 2

phia2 = np.deg2rad(10) # phase shift of amplitude modulation for oscillation 2
phia3 = np.deg2rad(90) # phase shift of amplitude modulation for oscillation 2

## Simulate Signals

In [ ]:
time = np.linspace(0, t, int(fs*t), endpoint=False)

oscill1 = np.sin(2*np.pi*f*time) # sine wave at frequency f
oscill2 = np.sin(2*np.pi*f*time + phi2) # wave at frequency f
oscill3 = np.sin(2*np.pi*f*time + phi3) # wave at frequency f

# noise parameters
mu = 0 # noise mean
sigma = 0.5 # noise std
noise1 = np.random.normal(mu, sigma, len(time))
noise2 = np.random.normal(mu, sigma, len(time))
noise3 = np.random.normal(mu, sigma, len(time))

# amplitude
A1 = np.sin(2*np.pi*fa*time)
A2 = np.sin(2*np.pi*fa*time + phia2)
A3 = np.sin(2*np.pi*fa*time + phia3)

signal1 = oscill1*A1 + noise1
signal2 = oscill2*A2 + noise2
signal3 = oscill3*A3 + noise3

## Plot the Simulated Data

In [ ]:
# plot the simulated data
fig1, ax = plt.subplots()
ax.plot(time,signal1)
ax.plot(time,signal2)
ax.plot(time,signal3)
ax.set(xlabel='time (s)', title='%dHz sine waves with random noise'%(f))

ax.set(xlim=(0, 0.5))

## Create multiple epochs of simulated data by changing the noise

In [ ]:
simdata = []

for epoc in range(20):
    noise1 = np.random.normal(mu, sigma, len(time))
    noise2 = np.random.normal(mu, sigma, len(time))
    noise3 = np.random.normal(mu, sigma, len(time))

    signal1 = oscill1*A1 + noise1
    signal2 = oscill2*A2 + noise2
    signal3 = oscill3*A3 + noise3
    
    simdata.append(np.array([signal1,signal2,signal3]))
  

## Compute the connectivity Matrices from the Simulated Signals

In [ ]:
  

#%%
indices = ([0, 0], [1, 2])

conn = []

for method in ["pli", "wpli", "dpli"]:
    conn.append(
        spectral_connectivity_epochs(
            simdata,
            method=method,
            sfreq=fs,
            indices=indices,
            fmin=45,
            fmax=55,
            faverage=True,
        ).get_data()[:, 0]
    )
conn = np.array(conn)


###############################################################################
# The estimated connectivites are shown in the figure below, which provides
# insight into the differences between PLI/wPLI, and dPLI.
#
#
# **Similarities Of All Measures**
#
# * Capture presence of connectivity in same situations (phase difference of
#   :math:`\pm\frac{\pi}{2}`)
# * Do not predict connectivity when phase difference is a multiple of
#   :math:`\pi`
# * Bounded between :math:`0` and :math:`1`
#
# **How dPLI is Different Than PLI/wPLI**
#
# * Null connectivity is :math:`0` for PLI and wPLI, but :math:`0.5` for dPLI
# * dPLI differentiates whether the reference signal is leading or lagging the
#   other signal (lagging if :math:`0 <= dPlI < 0.5`, leading if
#   :math:`0.5 < dPLI <= 1.0`)

## Plot the Phase Based Connnectivity Matrices

In [ ]:
x = np.arange(2)

plt.figure()
plt.bar(x - 0.2, conn[0], 0.2, align="center", label="PLI")
plt.bar(x, conn[1], 0.2, align="center", label="wPLI")
plt.bar(x + 0.2, conn[2], 0.2, align="center", label="dPLI")

plt.title("Connectivity Estimation Comparison")
plt.xticks(x, (r"$\pi/2$", r"$\pi/18$"))
plt.legend()
plt.xlabel("Phase Difference")
plt.ylabel("Estimated Connectivity")

plt.show()

## Filter and plot the data to prepare the data for envelope correlation

In [ ]:
#%% Envelope correlation
l_freq = 2
h_freq = 55
filtered_data = mne.filter.filter_data(simdata, fs, l_freq, h_freq)

n_del = int(fs/4)
timekeep = np.ones(time.shape)
timekeep[0:n_del] = 0
timekeep[-n_del:len(time)] = 0
timef = time[timekeep==1]

filtered_data = filtered_data[:,:,timekeep==1]

# plot the simulated data
ep = 5 # epoch to plot
fig1, ax = plt.subplots()
ax.plot(timef,filtered_data[ep,0,:])
ax.plot(timef,filtered_data[ep,1,:])
ax.plot(timef,filtered_data[ep,2,:])
ax.set(xlabel='time (s)', title='%d-%dHz filtered simulated data'%(l_freq,h_freq))
ax.set(xlim=(1, 1.5))

## Compute and display the envelope correlation matrices with and without Orthogonalization

In [ ]:
envcon = envelope_correlation(filtered_data, 
                     names=None, 
                     orthogonalize=False, 
                     log=False, 
                     absolute=True, 
                     verbose=None)
# Average over epochs
envcon = envcon.combine()
envcon = envcon.get_data(output="dense")[:, :, 0]


envcon_ortho = envelope_correlation(filtered_data, 
                     names=None, 
                     orthogonalize='pairwise', 
                     log=False, 
                     absolute=True, 
                     verbose=None)
# Average over epochs
envcon_ortho = envcon_ortho.combine()
envcon_ortho = envcon_ortho.get_data(output="dense")[:, :, 0]

def plot_corr(corr, title):
    fig, ax = plt.subplots(figsize=(4, 4), constrained_layout=True)
    im = ax.imshow(corr, cmap="viridis", clim=[0, 0.7])  # clim=np.percentile(corr, [5, 95])
    fig.colorbar(im, orientation='vertical')
    fig.suptitle(title)


plot_corr(envcon, "Envelope correlation  non-corrected")
plot_corr(envcon_ortho, "Envelope correlation with pairwise orthogonalization")